In [0]:
%sql
-- Step 1 (CTE): build the ranking, exactly like in step 9.3
WITH ranked AS (
    SELECT
        year,
        month,
        month_name,
        category_name,
        total_amount,
        -- Position of each category inside its own month, biggest spending first
        RANK() OVER (PARTITION BY year, month ORDER BY total_amount DESC) AS rank_in_month
    FROM workspace.gold.mart_monthly_category
    WHERE flow_type = 'outflow'          -- only spending categories
)
-- Step 2: now rank_in_month exists, so we can filter on it
SELECT year, month, month_name, category_name, total_amount, rank_in_month
FROM ranked
WHERE rank_in_month <= 2                 -- keep only the Top 2 of each month
ORDER BY year, month, rank_in_month

In [0]:
%sql
-- subquerie 

SELECT category_name, total_transactions, failed_transactions, failure_rate_pct
FROM workspace.gold.mart_failure_rate_category
WHERE failure_rate_pct > (
    -- Overall failure rate = total failed / total transactions.
    -- We do NOT use AVG(failure_rate_pct): it gives every category the same weight,
    -- so a tiny category like 'unknown' (2 transactions) would distort the result.
    -- SUM / SUM counts each transaction once, matching the fact table (162 / 500).
    SELECT ROUND(SUM(failed_transactions) / SUM(total_transactions) * 100, 2)
    FROM workspace.gold.mart_failure_rate_category
)
ORDER BY failure_rate_pct DESC

In [0]:
%sql
-- USO DE AVG em vez de SUM traz um resultado diferente!

SELECT category_name, total_transactions, failed_transactions, failure_rate_pct
FROM workspace.gold.mart_failure_rate_category
WHERE failure_rate_pct > (
    -- Scalar subquery: simple average of the category rates
    -- (every category weighs the same, whatever its size)
    SELECT AVG(failure_rate_pct)
    FROM workspace.gold.mart_failure_rate_category
)
ORDER BY failure_rate_pct DESC

In [0]:
%sql
-- CASE WHEN
-- Objective: how risky is each category? Classify its failure rate as high, medium or low.
SELECT
    category_name,
    failure_rate_pct,
    -- CASE WHEN works like an if / elif / else: the first true condition wins
    CASE
        WHEN failure_rate_pct >= 35 THEN 'high'
        WHEN failure_rate_pct >= 30 THEN 'medium'
        ELSE 'low'
    END AS risk_level
FROM workspace.gold.mart_failure_rate_category
ORDER BY failure_rate_pct DESC

In [0]:
%sql
-- HAVING
-- Objective: which categories have 30 or more failed transactions?

SELECT
    c.category_name,
    COUNT(*) AS failed_count
FROM workspace.gold.fact_transactions f
JOIN workspace.gold.dim_category c ON f.category_key = c.category_key
WHERE f.status = 'failed'          -- WHERE filters ROWS, before grouping
GROUP BY c.category_name
HAVING COUNT(*) >= 30              -- HAVING filters GROUPS, after grouping
ORDER BY failed_count DESC